In [134]:
import pandas as pd
pd.set_option('display.max_columns', None)
caminho_do_arquivo = r"..\..\Data\V_OCORRENCIA_AMPLA.json"
df = pd.read_json(caminho_do_arquivo, encoding='utf-8-sig')

colunas = ["Numero_da_Ocorrencia", "Classificacao_da_Ocorrência", "Data_da_Ocorrencia","Municipio","UF","Regiao","Nome_do_Fabricante","Modelo"]
df = df[colunas]
df.rename( columns={  'Classificacao_da_Ocorrência' : 'Classificacao_da_Ocorrencia'  } ,inplace=True )

df.head(3)

,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante,Modelo
0,7762,Incidente,2018-03-21,SÃO PAULO,SP,Sudeste,AGUSTA,AW109SP
1,7759,Acidente,2018-03-14,MONTES CLAROS,MG,Sudeste,CESSNA AIRCRAFT,A152
2,7758,Acidente,2018-01-26,INACIOLÂNDIA,GO,Centro-Oeste,CESSNA AIRCRAFT,T188C


In [135]:
df.dtypes

Numero_da_Ocorrencia           int64
Classificacao_da_Ocorrencia      str
Data_da_Ocorrencia               str
Municipio                        str
UF                               str
Regiao                           str
Nome_do_Fabricante               str
Modelo                           str
dtype: object

In [136]:
df['Data_da_Ocorrencia'] = pd.to_datetime(df['Data_da_Ocorrencia'])
df.dtypes

Numero_da_Ocorrencia                    int64
Classificacao_da_Ocorrencia               str
Data_da_Ocorrencia             datetime64[us]
Municipio                                 str
UF                                        str
Regiao                                    str
Nome_do_Fabricante                        str
Modelo                                    str
dtype: object

https://docs.python.org/pt-br/3.10/library/datetime.html

In [137]:
from datetime import datetime
ano_atual = datetime.now().year
print(ano_atual)


2026


In [138]:
ano_atual = 2018
df = df[df['Data_da_Ocorrencia'].dt.year == ano_atual]
df.head(10)

,Numero_da_Ocorrencia,Classificacao_da_Ocorrencia,Data_da_Ocorrencia,Municipio,UF,Regiao,Nome_do_Fabricante,Modelo
0,7762,Incidente,2018-03-21,SÃO PAULO,SP,Sudeste,AGUSTA,AW109SP
1,7759,Acidente,2018-03-14,MONTES CLAROS,MG,Sudeste,CESSNA AIRCRAFT,A152
2,7758,Acidente,2018-01-26,INACIOLÂNDIA,GO,Centro-Oeste,CESSNA AIRCRAFT,T188C
3,7758,Acidente,2018-01-26,INACIOLÂNDIA,GO,Centro-Oeste,CESSNA AIRCRAFT,T188C
4,7757,Incidente Grave,2018-03-18,TORRES,RS,Sul,PIPER AIRCRAFT,PA-34-200
7,7756,Incidente Grave,2018-03-18,IGARASSÚ,PE,Nordeste,KAIO AMARAL RANGEL,RV-7
8,7755,Incidente Grave,2018-03-18,MARAGOGI,AL,Nordeste,HENRIQUE DRUMOND LIMA DE OLIVEIRA,PARADISE P-1
9,7754,Incidente Grave,2018-03-17,BRAGANÇA PAULISTA,SP,Sudeste,NEIVA,56-C
10,7754,Incidente Grave,2018-03-17,BRAGANÇA PAULISTA,SP,Sudeste,NEIVA,56-C
11,7753,Acidente,2018-03-16,AVANHANDAVA,SP,Sudeste,NEIVA,EMB-201A


In [ ]:

import pandas as pd
from sqlalchemy import create_engine ,Integer, String, Date,VARCHAR,text

dbname   = 'postgres-anac'
user     = 'postgres'
password = '432931'
host     = 'immich-server.tailab6cca.ts.net'
port     = '5432' 

conexao_str = f'postgresql://{user}:{password}@{host}:{port}/{dbname}'
engine = create_engine(conexao_str)

nome_tabela = 'anac' 

# Deletar registros com base no ano atual
cursor=engine.connect() 
delete = text(f'delete from anac where extract(year from "Data_da_Ocorrencia")= {ano_atual}')
cursor.execute(delete)
cursor.commit()

# Enviar DataFrame para o banco de dados
df.to_sql(nome_tabela, engine, index=False, if_exists='append',
                     dtype={ 
                           'Numero_da_Ocorrencia' :   Integer ,
                           'Classificacao_da_Ocorrencia': VARCHAR(50),
                           'Data_da_Ocorrencia':Date,
                           'Municipio': VARCHAR(100),
                           'UF': VARCHAR(20),
                           'Regiao': VARCHAR(50),
                           'Nome_do_Fabricante': VARCHAR(100)
                           })

cursor.close()
engine.dispose()